# AuditMind — Slither vs Claude Haiku

Lane-1 benchmark on a frozen corpus of 723 files (212 vulnerable / 511 clean), built and hashed
in `01_data_engineering.ipynb`. No preprocessing happens here.

**Stage 2 only.** Both tools run once; raw output goes to disk unparsed. Detector names are
mapped to categories at *scoring* time (Stage 3), so the mapping stays revisable without
rescanning or re-spending.

**One-time prerequisites** (terminal, not here):

```bash
curl -L https://foundry.paradigm.xyz | bash
export PATH="$PATH:/Users/mk/.foundry/bin"
foundryup
echo 'export PATH="$PATH:/Users/mk/.foundry/bin"' >> ~/.zshrc
```

Foundry is required because several clean-pool repos (OpenZeppelin, Seaport, Aave-v3) ship a
`foundry.toml`, which Slither auto-detects and then shells out to `forge`.

## Setup

In [1]:
import os, sys, json, re, time, hashlib, subprocess
from pathlib import Path
from collections import Counter
import pandas as pd

PROJECT_ROOT = "/Users/mk/Desktop/mk/AuditMind"
os.chdir(PROJECT_ROOT)

# VS Code's kernel does not always inherit the shell PATH
for extra in ["/Users/mk/.foundry/bin"]:
    if extra not in os.environ["PATH"]:
        os.environ["PATH"] = f"{extra}:{os.environ['PATH']}"

STAGE2_DIR = Path("data/stage2_results")
STAGE2_DIR.mkdir(parents=True, exist_ok=True)

INCLUDE_OZ = os.path.abspath("node_modules_oz/node_modules")
INCLUDE_OZ_V4 = os.path.abspath("node_modules_oz_v4/node_modules")

# Ground-truth spelling. Note denial_of_service, NOT dos_gas_limit.
CATEGORIES = ["reentrancy", "access_control", "unchecked_low_level_calls",
              "denial_of_service", "time_manipulation"]

print("cwd:", os.getcwd())

cwd: /Users/mk/Desktop/mk/AuditMind


## Verify frozen corpus

In [2]:
corpus_record = json.load(open("data/manifests/frozen_corpus_record.json"))
stage1_manifest = json.load(open("data/manifests/stage1_manifest.json"))

def canonical_json(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"))

recomputed = hashlib.sha256(
    canonical_json(sorted(stage1_manifest, key=lambda r: r["path"])).encode()
).hexdigest()

assert recomputed == corpus_record["corpus_hash"], "CORPUS MISMATCH - stop"
assert len(stage1_manifest) == 723
print("Corpus verified\n")
print(json.dumps(corpus_record, indent=2))

Corpus verified

{
  "corpus_hash": "46f0c6bbb113a6dbb2b4ffba5fef1a35463fe673b04416d36290658a9535db56",
  "total_files": 723,
  "vulnerable_count": 212,
  "clean_count": 511,
  "sources": {
    "smartbugs": 116,
    "swc": 13,
    "dappscan": 83,
    "clean": 511
  }
}


### Known exclusion — `bad_randomness`

11 vulnerable files (8 SmartBugs, 3 SWC) carry a `bad_randomness` label. It is not one of the
five scored categories: the original dataset filters kept six, and the narrowing to five happened
later during benchmark planning and was never applied back to the corpus.

They stay in the manifest — the corpus and its hash are untouched — and are excluded at scoring
time instead. Scoring them would be actively misleading rather than a clean miss, since Slither
has a `weak-prng` detector and Haiku would likely file these under `time_manipulation` (most use
`block.timestamp` or `blockhash` as an entropy source).

**Effective per-source vulnerable counts for the five scored categories are therefore SmartBugs
108 and SWC 10**, not 116 and 13.

In [3]:
def entry_categories(rec):
    if "categories" in rec:
        return list(rec["categories"])
    return [rec["category"]] if "category" in rec else []

EXCLUDED_CATEGORY = "bad_randomness"
excluded_paths = {
    r["path"] for r in stage1_manifest
    if EXCLUDED_CATEGORY in entry_categories(r)
}

json.dump(sorted(excluded_paths),
          open("data/manifests/scoring_exclusions_bad_randomness.json", "w"), indent=2)

print(f"{len(excluded_paths)} bad_randomness files flagged for scoring exclusion")
print(Counter(r["source"] for r in stage1_manifest if r["path"] in excluded_paths))

11 bad_randomness files flagged for scoring exclusion
Counter({'smartbugs': 8, 'swc': 3})


## Environment check

Fails loudly and specifically rather than dying 400 files into a run.

In [4]:
class EnvCheckError(Exception):
    pass

def check_command(name, cmd, fix):
    try:
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=15)
    except FileNotFoundError:
        raise EnvCheckError(f"[{name}] not found or broken shebang.\n  Fix: {fix}")
    if r.returncode != 0:
        raise EnvCheckError(f"[{name}] rc={r.returncode}\n  {r.stderr[:300]}\n  Fix: {fix}")
    return r

PIP = f"{sys.executable} -m pip install --force-reinstall --no-deps"

check_command("solc-select", ["solc-select", "versions"], f"{PIP} solc-select")
check_command("slither", ["slither", "--version"], f"{PIP} slither-analyzer")
check_command("forge", ["forge", "--version"],
              "curl -L https://foundry.paradigm.xyz | bash && foundryup")

NEEDED_SOLC = ["0.4.26", "0.5.17", "0.6.12", "0.7.6",
               "0.8.20", "0.8.22", "0.8.24", "0.8.26", "0.8.27", "0.8.28"]
installed = check_command("solc-select versions", ["solc-select", "versions"], "").stdout
missing = [v for v in NEEDED_SOLC if v not in installed]
if missing:
    raise EnvCheckError(f"Missing solc: {missing}\n  Fix: solc-select install {' '.join(missing)}")

subprocess.run(["solc-select", "use", "0.8.20"], capture_output=True, text=True)
check_command("solc", ["solc", "--version"], "solc-select use 0.8.20")

for label, path in [("OpenZeppelin v5", INCLUDE_OZ), ("OpenZeppelin v4", INCLUDE_OZ_V4)]:
    if not os.path.exists(path):
        raise EnvCheckError(f"{label} missing at {path} - see 01_data_engineering.ipynb")

print("All environment checks passed.")

All environment checks passed.


## Path resolution

The flattened `data/{source}/` copies are Haiku's input only. Slither cannot use them: sibling
imports (`import "./interfaces/X.sol"`) do not resolve in a flat folder.

The flattened basename encodes the real nested path with `__` as the separator, so
`basename.replace("__", "/")` recovers the original location. `--base-path` then has to point at
the **repo root** (first path segment), not the file's own folder, or `../` imports fail.

SmartBugs and SWC are standalone single files, used as-is.

In [5]:
def resolve_scan_target(entry):
    """Returns (path_relative_to_base, base_dir) for compiling or scanning this entry."""
    if entry["source"] in ("dappscan", "clean"):
        real_rel = os.path.basename(entry["path"]).replace("__", "/")
        if os.path.exists(real_rel):
            repo_root = real_rel.split("/")[0]
            return os.path.relpath(real_rel, repo_root), os.path.abspath(repo_root)
    p = os.path.abspath(entry["path"])
    return os.path.basename(p), os.path.dirname(p)

resolved = sum(1 for r in stage1_manifest
               if os.path.exists(os.path.join(*resolve_scan_target(r)[::-1])))
print(f"{resolved}/{len(stage1_manifest)} resolve to a scannable location")

723/723 resolve to a scannable location


## Compile check

Establishes a known-good compiler per file, saved to `compiler_map.json` so the Slither run does
not have to rediscover it.

Three things this handles that a naive version pick does not:

- **Flag gating.** `--base-path` exists only from 0.6.x, `--include-path` only from 0.8.x.
  Passing either to 0.4/0.5 is an immediate `unrecognised option` error that breaks every file in
  those series.
- **Version escalation.** A file's *imports* can demand a newer pragma than the entry file
  declares, so we start at the file's own series and try everything newer; within 0.8, newest first.
- **Two OpenZeppelin versions.** v5 dropped `SafeMath`, which older DAppSCAN contracts import, so
  v4 is tried as a fallback include path.

Skip this cell if `compiler_map.json` already exists and the corpus has not changed.

In [6]:
SERIES_ORDER = ["0.4", "0.5", "0.6", "0.7", "0.8"]
SOLC_LADDER = {
    "0.4": ["0.4.26"], "0.5": ["0.5.17"], "0.6": ["0.6.12"], "0.7": ["0.7.6"],
    "0.8": ["0.8.28", "0.8.27", "0.8.26", "0.8.24", "0.8.22", "0.8.20"],
}
PRAGMA_RE = re.compile(r"pragma\s+solidity\s+([^;]+);")

def extract_pragma(path):
    m = PRAGMA_RE.search(open(path, errors="ignore").read())
    return m.group(1).strip() if m else None

def series_of(pragma):
    if not pragma:
        return "0.8"
    m = re.search(r"(\d+)\.(\d+)", pragma)
    return f"{m.group(1)}.{m.group(2)}" if m else "0.8"

_active = {"v": None}
def ensure_solc(v):
    if _active["v"] != v:
        r = subprocess.run(["solc-select", "use", v], capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(f"solc-select use {v}: {r.stderr}")
        _active["v"] = v

def solc_flags(series, oz=None):
    flags = ["--allow-paths", "."]
    if series in ("0.6", "0.7", "0.8"):
        flags += ["--base-path", "."]
    if series == "0.8" and oz:
        flags += ["--include-path", oz]
    return flags

def compile_check(entry, timeout_s=30):
    fname, base_dir = resolve_scan_target(entry)
    pragma = entry.get("pragma") or extract_pragma(os.path.join(base_dir, fname))
    s0 = series_of(pragma)
    start = SERIES_ORDER.index(s0) if s0 in SERIES_ORDER else 4

    last_err = ""
    for s in SERIES_ORDER[start:]:
        for ver in SOLC_LADDER[s]:
            ensure_solc(ver)
            for oz in ([INCLUDE_OZ, INCLUDE_OZ_V4] if s == "0.8" else [None]):
                try:
                    r = subprocess.run(["solc", "--bin", fname] + solc_flags(s, oz),
                                       capture_output=True, text=True,
                                       timeout=timeout_s, cwd=base_dir)
                    if r.returncode == 0:
                        return {"path": entry["path"], "source": entry["source"],
                                "status": "ok", "compiler": ver, "stderr": ""}
                    last_err = r.stderr[-500:]
                except subprocess.TimeoutExpired:
                    last_err = "TIMEOUT"

    return {"path": entry["path"], "source": entry["source"],
            "status": "fail", "compiler": "", "stderr": last_err}

In [7]:
t0 = time.time()
compile_results = []
for i, r in enumerate(stage1_manifest):
    compile_results.append(compile_check(r))
    if (i + 1) % 100 == 0:
        print(f"{i+1}/{len(stage1_manifest)} - {time.time()-t0:.0f}s")

print(f"\nDONE in {time.time()-t0:.0f}s")
print(Counter(r["status"] for r in compile_results))
print(pd.DataFrame(compile_results).groupby(["source", "status"]).size().unstack(fill_value=0))

compiler_map = {r["path"]: r["compiler"] for r in compile_results if r["status"] == "ok"}
json.dump(compiler_map, open("data/manifests/compiler_map.json", "w"), indent=2)
print(f"\nsaved compiler_map.json ({len(compiler_map)} entries)")
print(Counter(compiler_map.values()))

100/723 - 30s
200/723 - 77s
300/723 - 113s
400/723 - 143s
500/723 - 172s
600/723 - 200s
700/723 - 235s

DONE in 240s
Counter({'ok': 723})
status      ok
source        
clean      511
dappscan    83
smartbugs  116
swc         13

saved compiler_map.json (723 entries)
Counter({'0.8.28': 480, '0.4.26': 158, '0.5.17': 43, '0.6.12': 37, '0.7.6': 5})


## Slither

Uses `compiler_map.json` directly instead of rediscovering versions.

`--json -` silently swallows error output when compilation fails — empty stdout *and* stderr with
returncode 1. So a non-JSON result triggers a rerun without `--json`, purely to capture the real
diagnostic text.

Stored per file: full `raw_stdout` (detector names, impact, confidence and `source_mapping.lines`
are all recoverable from it), `raw_stderr`, `wallclock_s`, `compiler`, `returncode`. Nothing is
mapped to a category here.

In [8]:
compiler_map = json.load(open("data/manifests/compiler_map.json"))

def run_slither(entry, timeout_s=120):
    fname, base_dir = resolve_scan_target(entry)
    ver = compiler_map.get(entry["path"])
    base = {"path": entry["path"], "source": entry["source"],
            "label_type": entry["label_type"], "compiler": ver}

    if ver is None:
        return {**base, "status": "no_compiler", "wallclock_s": 0,
                "returncode": None, "raw_stdout": "", "raw_stderr": ""}

    ensure_solc(ver)
    series = ".".join(ver.split(".")[:2])
    cmd = ["slither", fname, "--solc-args", " ".join(solc_flags(series, INCLUDE_OZ))]

    t0 = time.time()
    try:
        r = subprocess.run(cmd + ["--json", "-"], capture_output=True, text=True,
                           timeout=timeout_s, cwd=base_dir)
        elapsed = time.time() - t0
        if r.stdout.strip().startswith("{"):
            return {**base, "status": "ok", "wallclock_s": elapsed,
                    "returncode": r.returncode, "raw_stdout": r.stdout,
                    "raw_stderr": r.stderr[-5000:]}
        r2 = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout_s, cwd=base_dir)
        return {**base, "status": "error", "wallclock_s": elapsed,
                "returncode": r2.returncode, "raw_stdout": r2.stdout,
                "raw_stderr": r2.stderr[-5000:]}
    except subprocess.TimeoutExpired:
        return {**base, "status": "timeout", "wallclock_s": timeout_s,
                "returncode": None, "raw_stdout": "", "raw_stderr": "TIMEOUT"}

### Full Slither run

Do not interrupt. The assert guards the save, but interrupting still loses the in-memory results
and means starting over.

**Skip if `slither_results.json` already holds 723 entries** — the next cell loads it either way.

In [9]:
t0 = time.time()
slither_results = []
for i, r in enumerate(stage1_manifest):
    slither_results.append(run_slither(r))
    if (i + 1) % 50 == 0:
        print(f"{i+1}/{len(stage1_manifest)} - {time.time()-t0:.0f}s")

print(f"\nDONE in {time.time()-t0:.0f}s")
print(Counter(r["status"] for r in slither_results))

assert len(slither_results) == 723, f"got {len(slither_results)} - DO NOT SAVE, rerun"
json.dump(slither_results, open(STAGE2_DIR / "slither_results.json", "w"))
print(f"saved {len(slither_results)} results")
print(pd.DataFrame(slither_results).groupby(["source", "status"]).size().unstack(fill_value=0))

50/723 - 69s
100/723 - 129s
150/723 - 254s
200/723 - 396s
250/723 - 482s
300/723 - 528s
350/723 - 626s
400/723 - 694s
450/723 - 769s
500/723 - 823s
550/723 - 885s
600/723 - 944s
650/723 - 992s
700/723 - 1033s

DONE in 1066s
Counter({'ok': 678, 'error': 45})
saved 723 results
status     error   ok
source               
clean         26  485
dappscan      16   67
smartbugs      3  113
swc            0   13


## Slither failures

Files Slither could not analyze are excluded from scoring for **both** tools, so Haiku gets no
free credit on files its competitor never saw. All 723 compile, so these are Slither's own
analysis stage failing after a successful compile — which is a property of the tool, not a gap in
the setup.

In [10]:
slither_results = json.load(open(STAGE2_DIR / "slither_results.json"))

def failure_reason(r):
    e = r["raw_stderr"] + r["raw_stdout"]
    if r["status"] == "timeout":
        return "timeout (analysis exceeded 120s)"
    if "NotConstant" in e:
        return "Slither constant-folding limitation"
    if "AssertionError" in e:
        return "Slither IR assertion failure"
    if "Unknown file:" in e or "not found: File not found" in e:
        return "crytic-compile import resolution (stricter than solc)"
    if "Base constructor parameters" in e:
        return "OZ v5 constructor signature mismatch"
    if "Failed to generate IR" in e:
        return "IR generation failure"
    return "other"

failures = [{**r, "failure_reason": failure_reason(r)}
            for r in slither_results if r["status"] != "ok"]

print(Counter(r["failure_reason"] for r in failures))
print()
print(pd.DataFrame(failures).groupby(["source", "failure_reason"]).size().unstack(fill_value=0))

json.dump([{k: r[k] for k in ("path", "source", "label_type", "status", "failure_reason")}
           for r in failures],
          open(STAGE2_DIR / "slither_failures.json", "w"), indent=2)

slither_ok_paths = {r["path"] for r in slither_results if r["status"] == "ok"}
print(f"\n{len(slither_ok_paths)} analyzed, {len(failures)} excluded")

Counter({'crytic-compile import resolution (stricter than solc)': 35, 'OZ v5 constructor signature mismatch': 5, 'Slither IR assertion failure': 2, 'other': 2, 'Slither constant-folding limitation': 1})

failure_reason  OZ v5 constructor signature mismatch  \
source                                                 
clean                                              0   
dappscan                                           5   
smartbugs                                          0   

failure_reason  Slither IR assertion failure  \
source                                         
clean                                      0   
dappscan                                   0   
smartbugs                                  2   

failure_reason  Slither constant-folding limitation  \
source                                                
clean                                             0   
dappscan                                          0   
smartbugs                                         1   

## Haiku

Runs on the files Slither analyzed, so both tools are measured on the same set.

Per the locked design: temperature 0, three calls per file, majority vote per category. All three
raw response texts are stored unparsed alongside the parsed verdicts, plus per-call token counts
and wallclock. A parse failure is retried once and then logged as `parse_error` — never coerced
to all-false, which would inflate false negatives for the wrong reason.

The prompt is a single fixed general-audit prompt, written before any results were seen and
reproduced verbatim below. No retrieval, no few-shot, no per-category tuning. It deliberately
carries **no hint about incomplete or extracted files**: Slither cannot receive an equivalent
hint, and how each tool fails on truncated DAppSCAN files is itself a result.

In [ ]:
from getpass import getpass
import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("ANTHROPIC_API_KEY: ")

client = anthropic.Anthropic()
HAIKU_MODEL = "claude-haiku-4-5-20251001"
MAX_TOKENS = 1000

SYSTEM_PROMPT = """You are a Solidity security analyzer. Analyze the contract below for exactly
five vulnerability categories. Report only what is present in this code.

The contract's pragma is shown near the top. Judge exploitability against that version. Some
patterns that are vulnerable in 0.4.x are safe by default in 0.8.x.

CATEGORIES

reentrancy
  An external call transfers control to another contract before the caller's state is updated,
  allowing the callee to re-enter and observe or exploit stale state. Look for state changes
  after .call/.send/.transfer or calls to arbitrary addresses.
  Not this: .transfer() or .send() with the 2300 gas stipend, or a function already protected by
  a reentrancy guard modifier.

access_control
  A function that changes ownership, moves funds, or alters critical state is callable by an
  unauthorized party. Includes missing modifiers, wrong visibility, unprotected initializers,
  tx.origin authentication, and unprotected selfdestruct or delegatecall.
  Not this: a view or pure function without a modifier, or a function that is intentionally
  public and changes only the caller's own state.

unchecked_low_level_calls
  The return value of .call, .delegatecall, .staticcall, or .send is not checked, so a failed
  call proceeds silently as if it succeeded.
  Not this: a call whose return value is checked by require, assert, or an if that reverts.

denial_of_service
  Execution can be blocked or made unaffordable. Includes unbounded loops over arrays that grow
  with user input, and functions whose progress depends on an external call that a participant
  can force to fail.
  Not this: a loop bounded by a fixed constant, or one over an array only the owner can extend.

time_manipulation
  block.timestamp or block.number is used for logic a miner can profitably influence, such as
  randomness, deadlines, payout eligibility, or lock expiry.
  Not this: block.timestamp used for an event log, or for a deadline measured in days where a
  few seconds of miner drift is irrelevant.

OUTPUT

Return a single JSON object and nothing else. No prose, no markdown fences. All five keys must
be present every time.

{
  "reentrancy": {"found": true, "lines": [42]},
  "access_control": {"found": false, "lines": []},
  "unchecked_low_level_calls": {"found": true, "lines": [88, 91]},
  "denial_of_service": {"found": false, "lines": []},
  "time_manipulation": {"found": false, "lines": []}
}

"found" is true or false. "lines" is a list of integers, empty when found is false, holding the
line where the vulnerability is triggered. Several genuinely separate findings in one category
are fine; do not list every line a single finding touches. Report a category only if you can
point to a specific line."""

print(f"{HAIKU_MODEL} | temp=0 | system prompt {len(SYSTEM_PROMPT)} chars")

### Line numbering

Stage 1 never prepended line numbers, so the files in `stage1_preprocessed/` are unnumbered.
Without them Haiku is guessing at line references and the localization sweep in Stage 3 would be
meaningless.

Numbering happens in memory at call time rather than rewriting 723 files on disk, so the frozen
corpus and its hash stay untouched.

In [ ]:
def numbered_source(stage1_path):
    lines = open(stage1_path, errors="ignore").read().split("\n")
    return "\n".join(f"{i}: {ln}" for i, ln in enumerate(lines, start=1))

_probe = [r for r in stage1_manifest if r["path"] in slither_ok_paths][0]
print(numbered_source(_probe["stage1_path"])[:400])

In [ ]:
import random

# 50 vulnerable + 50 clean rather than proportional. A proportional sample lands ~70% clean,
# and stability on a clean file (expected answer: nothing found) is likely higher than on a file
# where the model has to identify something - which would understate disagreement.
N_VULN, N_CLEAN = 50, 50
SEED = 20260901

targets = [r for r in stage1_manifest if r["path"] in slither_ok_paths]
print(f"{len(targets)} target files")

rng = random.Random(SEED)
vuln = [r for r in targets if r["label_type"] == "vulnerable"]
clean = [r for r in targets if r["label_type"] == "clean"]

triplicate = []
by_source = {}
for r in vuln:
    by_source.setdefault(r["source"], []).append(r)
for s, group in sorted(by_source.items()):          # stratify vulnerable by source
    k = min(max(round(N_VULN * len(group) / len(vuln)), 1), len(group))
    triplicate += rng.sample(sorted(group, key=lambda r: r["path"]), k)
triplicate += rng.sample(sorted(clean, key=lambda r: r["path"]), min(N_CLEAN, len(clean)))

triplicate_paths = {r["path"] for r in triplicate}
single_paths = {r["path"] for r in targets} - triplicate_paths

print(f"triplicate: {len(triplicate_paths)} files x 3 calls")
print(f"single:     {len(single_paths)} files x 1 call")
print(f"total requests: {len(triplicate_paths)*3 + len(single_paths)}")
print()
for key, n in sorted(Counter((r["source"], r["label_type"]) for r in triplicate).items()):
    print(f"  {key[0]:10} {key[1]:10} {n}")

### Full Haiku run

669 files at three calls each is roughly 2,000 requests. The system block is identical every
time, so prompt caching applies.

Results are written incrementally, so an interruption here costs only the remaining files rather
than the whole run.

In [ ]:
targets = [r for r in stage1_manifest if r["path"] in slither_ok_paths]
print(f"{len(targets)} files x {N_CALLS} calls = {len(targets)*N_CALLS} requests")

partial_path = STAGE2_DIR / "haiku_results_partial.json"
haiku_results = json.load(open(partial_path)) if partial_path.exists() else []
done = {r["path"] for r in haiku_results}
remaining = [r for r in targets if r["path"] not in done]
print(f"{len(done)} already done, {len(remaining)} remaining")

t0 = time.time()
for i, entry in enumerate(remaining):
    try:
        haiku_results.append(run_haiku(entry))
    except Exception as e:
        haiku_results.append({"path": entry["path"], "source": entry["source"],
                              "label_type": entry["label_type"], "status": "api_error",
                              "error": str(e)[:500], "majority_verdict": None,
                              "raw_verdicts": [], "raw_texts": [],
                              "disagreement_rate": None, "input_tokens": 0,
                              "output_tokens": 0, "wallclock_s": 0, "n_parse_failures": 0})
    if (i + 1) % 25 == 0:
        json.dump(haiku_results, open(partial_path, "w"))
        print(f"{i+1}/{len(remaining)} - {time.time()-t0:.0f}s")

json.dump(haiku_results, open(STAGE2_DIR / "haiku_results.json", "w"))
print(f"\nDONE in {time.time()-t0:.0f}s")
print(Counter(r["status"] for r in haiku_results))
print(f"saved {len(haiku_results)} results")

## Haiku run summary

In [ ]:
haiku_results = json.load(open(STAGE2_DIR / "haiku_results.json"))
ok = [r for r in haiku_results if r["status"] == "ok"]

tot_in = sum(r["input_tokens"] for r in haiku_results)
tot_out = sum(r["output_tokens"] for r in haiku_results)
tot_s = sum(r["wallclock_s"] for r in haiku_results)

print(f"analyzed        {len(ok)}/{len(haiku_results)}")
print(f"parse errors    {sum(1 for r in haiku_results if r['status'] == 'parse_error')}")
print(f"api errors      {sum(1 for r in haiku_results if r['status'] == 'api_error')}")
print(f"single-call parse failures  {sum(r['n_parse_failures'] for r in haiku_results)}")
print()
print(f"tokens   in {tot_in:,}  out {tot_out:,}   (3x majority)")
print(f"         in {tot_in//N_CALLS:,}  out {tot_out//N_CALLS:,}   (1x equivalent)")
print(f"wallclock {tot_s:.0f}s total, {tot_s/max(len(haiku_results),1):.2f}s/file")

rates = [r["disagreement_rate"] for r in ok if r["disagreement_rate"] is not None]
if rates:
    print(f"\n3-run disagreement: mean {sum(rates)/len(rates):.3f}, "
          f"{sum(1 for x in rates if x > 0)}/{len(rates)} files with any disagreement")

flag_counts = Counter()
for r in ok:
    for c, v in r["majority_verdict"].items():
        if v["found"]:
            flag_counts[c] += 1
print("\nflags by category (majority verdict):")
for c in CATEGORIES:
    print(f"  {c:28} {flag_counts[c]}")

---

Stage 2 is complete once both `slither_results.json` and `haiku_results.json` are on disk.

Stage 3 loads them from disk and applies scoring — detector-to-category mapping, the type-aware
TP/FP/FN/TN criterion, the line-tolerance sweep on SmartBugs, and the `bad_randomness` and
Slither-failure exclusions. Nothing in Stage 3 re-triggers a scan or an API call.